# DATA LOAD AND CLEANING

In [70]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [71]:
df = pd.read_csv(r"C:\Users\User\airline-sentiment\data\Airline Review.csv")
df.head()

,Unnamed: 0,Airline Name,Overall_Rating,Review_Title,Review Date,Verified,Review,Aircraft,Type Of Traveller,Seat Type,Route,Date Flown,Seat Comfort,Cabin Staff Service,Food & Beverages,Ground Service,Inflight Entertainment,Wifi & Connectivity,Value For Money,Recommended
0,0,AB Aviation,9,"""pretty decent airline""",11th November 2019,True,Moroni to Moheli. Turned out to be a pretty ...,NaN,Solo Leisure,Economy Class,Moroni to Moheli,November 2019,4.0,5.0,4.0,4.0,NaN,NaN,3.0,yes
1,1,AB Aviation,1,"""Not a good airline""",25th June 2019,True,Moroni to Anjouan. It is a very small airline...,E120,Solo Leisure,Economy Class,Moroni to Anjouan,June 2019,2.0,2.0,1.0,1.0,NaN,NaN,2.0,no
2,2,AB Aviation,1,"""flight was fortunately short""",25th June 2019,True,Anjouan to Dzaoudzi. A very small airline an...,Embraer E120,Solo Leisure,Economy Class,Anjouan to Dzaoudzi,June 2019,2.0,1.0,1.0,1.0,NaN,NaN,2.0,no
3,3,Adria Airways,1,"""I will never fly again with Adria""",28th September 2019,False,Please do a favor yourself and do not fly wi...,NaN,Solo Leisure,Economy Class,Frankfurt to Pristina,September 2019,1.0,1.0,NaN,1.0,NaN,NaN,1.0,no
4,4,Adria Airways,1,"""it ruined our last days of holidays""",24th September 2019,True,Do not book a flight with this airline! My fr...,NaN,Couple Leisure,Economy Class,Sofia to Amsterdam via Ljubljana,September 2019,1.0,1.0,1.0,1.0,1.0,1.0,1.0,no


In [72]:
print(f"\nHere is our dataframe's shape:{df.shape}\n")
df.info()


Here is our dataframe's shape:(23171, 20)

<class 'pandas.DataFrame'>
RangeIndex: 23171 entries, 0 to 23170
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Unnamed: 0              23171 non-null  int64  
 1   Airline Name            23171 non-null  str    
 2   Overall_Rating          23171 non-null  str    
 3   Review_Title            23171 non-null  str    
 4   Review Date             23171 non-null  str    
 5   Verified                23171 non-null  bool   
 6   Review                  23171 non-null  str    
 7   Aircraft                7129 non-null   str    
 8   Type Of Traveller       19433 non-null  str    
 9   Seat Type               22075 non-null  str    
 10  Route                   19343 non-null  str    
 11  Date Flown              19417 non-null  str    
 12  Seat Comfort            19016 non-null  float64
 13  Cabin Staff Service     18911 non-null  float64
 14  Food 

### Inspect each column
Recorde = 23171 | dimension = 20, which is higher so we might need to do feature elimination

In [73]:
#Unnamed: 0 is useless column so remove it
df.drop(columns=["Unnamed: 0"], inplace=True)
#Aircraft also useless for analysis and 7129 non-null values
df.drop(columns=["Aircraft"], inplace=True)

In [74]:
#str -> numeric
df["Overall_Rating"] = df["Overall_Rating"].astype(str).str.extract('(\d+)')
df["Overall_Rating"] = pd.to_numeric(df["Overall_Rating"])
df["Overall_Rating"].dtype

dtype('float64')

### Null values
There is a massive sum of null values in each column which "Wifi & Connectivity" = 17,251 which is 70 percentage of total records so dropping wouldn't be the smart choice

In [75]:
print(f"\nThis is our total number of null values in this entire dataset: {df.isnull().sum().sum()}\n")
df.isna().sum().sort_values(ascending=False)


This is our total number of null values in this entire dataset: 65796



Wifi & Connectivity       17251
Inflight Entertainment    12342
Food & Beverages           8671
Ground Service             4793
Cabin Staff Service        4260
Seat Comfort               4155
Route                      3828
Date Flown                 3754
Type Of Traveller          3738
Seat Type                  1096
Value For Money            1066
Overall_Rating              842
Airline Name                  0
Review                        0
Review_Title                  0
Review Date                   0
Verified                      0
Recommended                   0
dtype: int64

### Drop invalid rows

In [76]:
df = df.dropna(subset=["Overall_Rating", "Review"])

### Create sentiment labels


In [77]:
def sentiment_flag(r):
    if r <= 4:
        return "negative"
    elif r <= 6:
        return "neutral"
    else:
        return "positive"

df["sentiment"] = df["Overall_Rating"].apply(sentiment_flag)
df["sentiment"].value_counts()

sentiment
negative    16106
positive     4718
neutral      1505
Name: count, dtype: int64

### Encode Labels

In [78]:
label_map = {
    'negative': 0,
    'neutral': 1,
    'positive': 2
}
df['label'] = df['sentiment'].map(label_map)
df["label"].value_counts()

label
0    16106
2     4718
1     1505
Name: count, dtype: int64

### Combine text (Review text + Review title

In [79]:
df["full_review"] = df["Review_Title"].fillna('') +" "+ df["Review"].fillna('')
df["full_review"] = df["full_review"].str.strip() #This removes any accidental leading or trailing spaces
print(df["full_review"][0])

"pretty decent airline"   Moroni to Moheli. Turned out to be a pretty decent airline. Online booking worked well, checkin and boarding was fine and the plane looked well maintained. Its a very short flight - just 20 minutes or so so i didn't expect much but they still managed to hand our a bottle of water and some biscuits which i though was very nice. Both flights on time.


### Clean Recommended (for later validation)

In [80]:
df['Recommended_encoded'] = df["Recommended"].map({
    'yes': 1,
    'no': 0
})

## **BASELINE MODEL**

### Split train test

In [83]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['full_review'], df['label'], test_size=0.2, random_state=42, stratify=df['label'])
print(X_train.shape)
print(y_train.shape)

(17863,)
(17863,)


### Vectorization text

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

### Logistic Regression

In [86]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)
y_pred

array([0, 0, 0, ..., 0, 0, 0], shape=(4466,))

### Evaluate

In [90]:
from sklearn.metrics import *
classification_report = classification_report(y_test, y_pred)
print(classification_report)

              precision    recall  f1-score   support

           0       0.85      0.92      0.89      3221
           1       0.43      0.03      0.06       301
           2       0.65      0.66      0.66       944

    accuracy                           0.81      4466
   macro avg       0.64      0.54      0.53      4466
weighted avg       0.78      0.81      0.78      4466

